# Retrieval Latency Benchmark

Runs **only** the retrieval phase (no LLM generation, no answer evaluation) for each
ablation config and records per-question `retrieval_s`.

* Query-entity extraction is **skipped** for questions already in the cache — pure Neo4j latency.
* Results are patched directly into the existing `ablation-results-*/` CSVs so `summary()` picks them up automatically.
* Use `max_workers=1` (default) for accurate serial latency numbers.

In [1]:
import os
import sys
import pickle
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase

sys.path.insert(0, str(Path("..").resolve()))
load_dotenv("../.env")

True

In [2]:
from graph.embedder import Embedder
from graph.kin_rag import StatelessCypherRetriever, AblationConfig, AblationEvaluator

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

DATASET_NAME = "2wiki"  # "musique" or "2wiki"
N_EVAL = 500            # number of questions to benchmark
MAX_WORKERS = 1         # 1 = serial (accurate latency); >1 = parallel (throughput)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
embedder = Embedder(cache_path=Path("cache/embeddings_cache.pkl"))

[Embedder] Loaded 828 cached embeddings


In [3]:
with open(f"ground_truth-{DATASET_NAME}.pkl", "rb") as f:
    ground_truth = pickle.load(f)

print(f"Loaded {len(ground_truth)} questions for {DATASET_NAME}")

Loaded 1000 questions for 2wiki


In [4]:
evaluator = AblationEvaluator(
    driver=driver,
    embedder=embedder,
    ground_truth=ground_truth,
    output_dir=f"ablation-results-{DATASET_NAME}",
    retriever_cls=StatelessCypherRetriever,
    query_entity_cache_path=Path(f"cache/query_entities-{DATASET_NAME}.pkl"),
)

[Evaluator] Loaded 1000 cached query entities


In [5]:
configs = {
    "naive_vector": AblationConfig(name="naive-vector", naive_vector=True, top_k_entities=30),
    "no_qa":        AblationConfig(name="no-query-aware", query_aware=False, max_steps=3, decay=0.7),
    "steps_1":      AblationConfig(name="steps-1",  max_steps=1, decay=0.7),
    "steps_2":      AblationConfig(name="steps-2",  max_steps=2, decay=0.7),
    "steps_3":      AblationConfig(name="steps-3",  max_steps=3, decay=0.7),
    "steps_4":      AblationConfig(name="steps-4",  max_steps=4, decay=0.7),
    # "decay_03":     AblationConfig(name="decay-0.3", max_steps=3, decay=0.3),
    # "decay_05":     AblationConfig(name="decay-0.5", max_steps=3, decay=0.5),
    # "decay_07":     AblationConfig(name="decay-0.7", max_steps=3, decay=0.7),
    # "decay_09":     AblationConfig(name="decay-0.9", max_steps=3, decay=0.9),
    # "decay_10":     AblationConfig(name="decay-1.0", max_steps=3, decay=1.0),
}

## Run benchmark

Run all configs, or just a subset — comment out the ones you don't need.

Each call patches `retrieval_s` in the existing CSV for that config.

In [6]:
CONFIGS_TO_RUN = list(configs.keys())

timing_results = {}
for key in CONFIGS_TO_RUN:
    timing_results[key] = evaluator.benchmark_retrieval(
        configs[key],
        n_eval=N_EVAL,
        max_workers=MAX_WORKERS,
        patch_csv=True,
    )


  [retrieval-only] [naive-vector] naive vector search, top_k=30


naive-vector:   0%|          | 0/500 [00:00<?, ?it/s]

naive-vector: 100%|██████████| 500/500 [02:42<00:00,  3.07it/s]



  [naive-vector] n=500  avg=0.325s  p50=0.270s  p95=0.600s
  Patched: ablation-results-2wiki/naive-vector.csv

  [retrieval-only] [no-query-aware] steps=3, decay=0.7, uniform, threshold=0.01


no-query-aware: 100%|██████████| 500/500 [03:41<00:00,  2.26it/s]


...


  [steps-2] n=500  avg=0.021s  p50=0.012s  p95=0.066s
  Patched: ablation-results-2wiki/steps-2.csv

  [retrieval-only] [steps-3] steps=3, decay=0.7, query-aware, threshold=0.01


steps-3: 100%|██████████| 500/500 [00:45<00:00, 11.06it/s]



  [steps-3] n=500  avg=0.090s  p50=0.040s  p95=0.349s
  Patched: ablation-results-2wiki/steps-3.csv

  [retrieval-only] [steps-4] steps=4, decay=0.7, query-aware, threshold=0.01


steps-4: 100%|██████████| 500/500 [02:32<00:00,  3.28it/s]


  [steps-4] n=500  avg=0.304s  p50=0.157s  p95=1.036s
  Patched: ablation-results-2wiki/steps-4.csv


## Summary table

In [8]:
import pandas as pd

rows = []
for key, df in timing_results.items():
    rows.append({
        "config": configs[key].name,
        "n": len(df),
        "avg_retrieval_s": df["retrieval_s"].mean(),
        "p50_retrieval_s": df["retrieval_s"].median(),
        "p95_retrieval_s": df["retrieval_s"].quantile(0.95),
        "max_retrieval_s": df["retrieval_s"].max(),
    })

summary_df = pd.DataFrame(rows).sort_values("avg_retrieval_s")
summary_df.style.format({
    "avg_retrieval_s": "{:.3f}",
    "p50_retrieval_s": "{:.3f}",
    "p95_retrieval_s": "{:.3f}",
    "max_retrieval_s": "{:.3f}",
})

,config,n,avg_retrieval_s,p50_retrieval_s,p95_retrieval_s,max_retrieval_s
2,steps-1,500,0.008,0.007,0.016,0.088
3,steps-2,500,0.021,0.012,0.066,0.270
4,steps-3,500,0.090,0.040,0.349,2.043
5,steps-4,500,0.304,0.157,1.036,3.235
0,naive-vector,500,0.325,0.270,0.600,10.403
1,no-query-aware,500,0.443,0.477,0.688,10.454
